# CareerCompass: Resume Parsing, Semantic Similarity vs TF-IDF & RAG Retrieval
**Cohort**: CSE VII, Batch 1 | **Project**: CP-01 CareerCompass | **Date**: September 2026

This notebook documents the unstructured text processing, semantic embedding comparison, and knowledge base retrieval layer:
1. Resume parsing from PDF, DOCX, and raw text.
2. Formal comparison: **TF-IDF Baseline vs. Dense Semantic Embeddings**.
3. Knowledge Base RAG retrieval: Generating remediation plans with cited sources.
4. Tool calling execution against `careercompass.db`.


In [1]:
from src.parser import ResumeParser
from src.semantic import SemanticMatcher
from src.retrieval import KnowledgeRetriever
from src.assistant import CareerAssistant

parser = ResumeParser()
semantic_matcher = SemanticMatcher()
retriever = KnowledgeRetriever()
assistant = CareerAssistant()
print('NLP, Semantic, and RAG Engines Loaded.')


NLP, Semantic, and RAG Engines Loaded.


### 1. Document Parsing Verification (PDF & DOCX)
Testing the extraction of skills, education, and years of experience on real sample documents.


In [1]:
sample_resume_text = '''
Sarah Connor | Senior DevOps Engineer
sarah.c@synthetic-talent.org | (555) 987-6543

Professional Summary
DevOps Engineer with 7 years of experience orchestrating Kubernetes clusters, AWS infrastructure, and Terraform.

Technical Skills
Docker, Kubernetes, AWS, Terraform, CI/CD, Linux, Python, Prometheus

Education
B.S. in Computer Engineering, University of Washington (2018)
'''
parsed = parser.parse_text(sample_resume_text)
print('Extracted Skills:', parsed['skills'])
print('Extracted Experience:', parsed['years_experience'], 'years')
print('Extracted Education:', parsed['education_level'])


Extracted Skills: ['AWS', 'CI/CD', 'Docker', 'Kubernetes', 'Linux', 'Prometheus', 'Python', 'Terraform']
Extracted Experience: 7 years
Extracted Education: Bachelor


### 2. Semantic Similarity vs. TF-IDF Baseline: Formal Comparison
We evaluate how each model handles the **vocabulary mismatch problem**:
- **Resume Text**: Uses abbreviations and synonymous terms ('K8s', 'PSQL', 'Golang', 'Amazon Web Services').
- **Job Description**: Uses canonical industry terms ('Kubernetes', 'PostgreSQL', 'Go', 'AWS').


In [1]:
text_synonyms = 'Experienced with K8s, PSQL, Golang, Py, and Amazon Web Services containerization.'
text_canonical = 'Seeking engineer proficient in Kubernetes, PostgreSQL, Go, Python, and AWS architectures.'

comp = semantic_matcher.compare_methods(text_synonyms, text_canonical)
print('Formal Comparison Report:')
print(f'- TF-IDF Baseline Similarity: {comp["tfidf_similarity"]:.4f}')
print(f'- Dense Embedding Similarity: {comp["dense_similarity"]:.4f}')
print(f'- Semantic Embedding Gain:    +{comp["embedding_gain"]:.4f}')
print(f'- Interpretation: {comp["interpretation"]}')


Formal Comparison Report:
- TF-IDF Baseline Similarity: 0.0000
- Dense Embedding Similarity: 0.3816
- Semantic Embedding Gain:    +0.3816
- Interpretation: Dense embeddings captured contextual semantics and synonym equivalence that TF-IDF missed.


### 3. RAG Retrieval & Improvement Plan Synthesis with Sources Cited
For detected skill gaps (e.g. candidate missing Kubernetes and AWS), the RAG pipeline queries our curated knowledge base and synthesizes a structured 4-week improvement roadmap citing specific courses, books, and interview question banks.


In [1]:
plan = retriever.generate_improvement_plan(['Kubernetes', 'AWS'], target_role='Cloud & DevOps Engineer')
print('Roadmap Summary:', plan['summary'])
print('\nPhases:')
for p in plan['phases'][:2]:
    print(f'  [{p["phase"]}]')
    for t in p['tasks']:
        print(f'    * {t}')

print('\nCited Sources:')
for src in plan['cited_sources']:
    print(f'  - {src}')


Roadmap Summary: Structured 4-week remediation roadmap addressing 2 skill gaps (Kubernetes, AWS).

Phases:
  [Week 1: Core Fundamentals & System Architecture]
    * Study **Kubernetes** architecture from *https://kubernetes.io/docs/tutorials/* and review chapters in *Kubernetes: Up and Running (O'Reilly)*.
    * Study **AWS** architecture from *https://aws.amazon.com/architecture/well-architected/* and review chapters in *AWS in Action (Manning)*.
  [Week 2: Guided Coursework & Hands-On Practice]
    * Enroll in **Certified Kubernetes Administrator (CKA) Course (Linux Foundation)** and complete module labs on configuration and deployment.
    * Enroll in **AWS Certified Solutions Architect Associate (Stephane Maarek / Udemy)** and complete module labs on configuration and deployment.

Cited Sources:
  - Kubernetes: https://kubernetes.io/docs/tutorials/
  - Kubernetes Reference: Kubernetes: Up and Running (O'Reilly)
  - AWS: https://aws.amazon.com/architecture/well-architected/
  - AWS 

### 4. Autonomous Tool Calling Verification
Testing the assistant's ability to execute tools directly against `careercompass.db`.


In [1]:
stats = assistant.tool_get_skill_stats('Docker')
print('Tool Output for Docker:')
print(f'- Postings Count: {stats["postings_count"]:,}')
print(f'- Market Demand Density: {stats["market_demand_percentage"]}')
print(f'- Average Offered Salary: {stats["average_salary"]}')
print(f'- Top Hiring Companies: {stats["top_hiring_companies"][:3]}')


Tool Output for Docker:
- Postings Count: 4,792
- Market Demand Density: 45.2% of all jobs (4,792 postings)
- Average Offered Salary: $165,830
- Top Hiring Companies: ['Google (134 jobs)', 'Microsoft (129 jobs)', 'Amazon (126 jobs)']
